# TLC Yellow Taxi Pipeline — Gold Mart Demo

Run this notebook inside the Spark master container or against the Spark Thrift Server.

**Pre-requisites:** `make demo-ingest` has been run for at least one month (e.g. 2023-01).

```bash
# Start the stack and run a demo month
make up && make init && make demo-ingest
```

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("tlc-demo")
    .master("spark://spark-master:7077")
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.iceberg.type", "rest")
    .config("spark.sql.catalog.iceberg.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.iceberg.warehouse", "s3a://warehouse/")
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.iceberg.s3.path-style-access", "true")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

## 1. Bronze row count and schema verification

In [ ]:
bronze = spark.table("iceberg.bronze.yellow_trips")
print(f"Bronze rows: {bronze.count():,}")
print(f"Columns: {len(bronze.columns)}")
bronze.select(
    "tpep_pickup_datetime", "fare_amount", "total_amount",
    "passenger_email_token", "_salt_version", "_ingested_at"
).show(5)

## 2. PDPA token verification — tokens are 64-char hex, raw PII absent

In [ ]:
from pyspark.sql import functions as F

token_stats = bronze.select(
    F.count("passenger_email_token").alias("tokenized_rows"),
    F.count("*").alias("total_rows"),
    (F.count("passenger_email_token") / F.count("*")).alias("coverage_rate"),
    F.avg(F.length("passenger_email_token")).alias("avg_token_length"),
).collect()[0]

print(f"Token coverage: {token_stats['coverage_rate']:.2%}")
print(f"Avg token length: {token_stats['avg_token_length']:.0f} chars (expected 64)")
assert "passenger_email" not in bronze.columns, "Raw PII must not be in Bronze!"
print("✓ Raw PII not present in Bronze — PDPA pattern verified")

## 3. Gold — Top 10 pickup zones by revenue (January 2023)

In [ ]:
zone_revenue = spark.table("iceberg.gold.fct_zone_revenue_monthly")

zone_revenue.filter(
    (F.year("pickup_month") == 2023) & (F.month("pickup_month") == 1)
).orderBy(F.desc("total_revenue")) \
 .select("zone_name", "borough", "trip_count", "total_revenue", "avg_tip_pct") \
 .show(10)

## 4. Gold — Daily trip volume and revenue trend

In [ ]:
daily = spark.table("iceberg.gold.fct_trips_daily")

daily.groupBy("pickup_date") \
     .agg(
         F.sum("trip_count").alias("total_trips"),
         F.sum("total_revenue").alias("total_revenue"),
         F.avg("avg_tip_pct").alias("avg_tip_pct"),
     ) \
     .orderBy("pickup_date") \
     .show(10)

## 5. Gold — fct_trips grain check (each trip_id is unique)

In [ ]:
fct = spark.table("iceberg.gold.fct_trips")
total = fct.count()
distinct = fct.select("trip_id").distinct().count()
print(f"Total rows: {total:,}")
print(f"Distinct trip_ids: {distinct:,}")
assert total == distinct, "trip_id must be unique — grain violation!"
print("✓ trip_id uniqueness confirmed")

## 6. Governance — tokenization audit coverage

In [ ]:
spark.table("iceberg.gold.fct_tokenization_audit") \
     .select("pickup_month", "salt_version", "total_trips",
             "tokenized_count", "token_coverage_rate", "distinct_passengers") \
     .orderBy("pickup_month") \
     .show()

## 7. Schema evolution — cross-year query (requires backfill)

In [ ]:
# After running: make demo-backfill
spark.sql("""
    SELECT
        year(tpep_pickup_datetime)   AS year,
        month(tpep_pickup_datetime)  AS month,
        COUNT(*)                     AS trips,
        AVG(congestion_surcharge)    AS avg_congestion,
        AVG(airport_fee)             AS avg_airport_fee
    FROM iceberg.bronze.yellow_trips
    GROUP BY 1, 2
    ORDER BY 1, 2
""").show(20)

## 8. Iceberg snapshot history (time-travel support)

In [ ]:
spark.sql("""
    SELECT snapshot_id, committed_at, operation, summary
    FROM iceberg.bronze.yellow_trips.history
    ORDER BY committed_at DESC
    LIMIT 10
""").show(truncate=80)

## 9. SCD2 — dim_taxi_zone_snapshot (exactly one current record per zone)

In [ ]:
scd2 = spark.table("iceberg.gold.dim_taxi_zone_snapshot")
current = scd2.filter("dbt_valid_to IS NULL").count()
total_zones = 265
print(f"Current SCD2 records (dbt_valid_to IS NULL): {current}")
print(f"Expected: {total_zones} (one per TLC zone)")
scd2.filter("dbt_valid_to IS NULL").select(
    "location_id", "zone_name", "borough", "dbt_valid_from"
).orderBy("location_id").show(10)

## 10. Pipeline audit — end-to-end latency

In [ ]:
spark.table("iceberg.ops.pipeline_audit") \
     .select("dag_id", "task_id", "layer", "status",
             "rows_out", "duration_sec", "started_at") \
     .orderBy(F.desc("started_at")) \
     .show(20, truncate=40)